### Следующая ячейка - блок импорта общих библиотек и формирования переменных
#### должна выполняться первой

In [1]:
import os # Отключаем попытку tqdm найти визуальный виджет IProgress 
#from tqdm.auto import tqdm
os.environ['TQDM_DISABLE'] = '1'

In [2]:
# ИМПОРТ БИБЛИОТЕК
import os
from collections import Counter
from pathlib import Path
import httpx
from dotenv import load_dotenv
import faiss
# ИМПОРТ ФУНКЦИИ chank_text() вынесенной из учебного примера в /src/ai_engineer_preproject/__init__.py
from ai_engineer_preproject import chunk_text, VectorStore, VecIdxSearch
# ФОРМИРОВАНИЕ ПЕРЕМЕННЫХ
# Загружаем переменные из .env
load_dotenv()
# Получаем путь к папке с документами (данные для LLM)
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT"))
DOCS_DIR = Path(os.getenv("OLLAMA_DOCS"))
OLLAMA_URL = os.getenv("OLLAMA_URL")
print(f"Корнева папка проекта:\n{PROJECT_ROOT}")
print(f"Папка с документами датасета:\n{DOCS_DIR}")
print(f"URL для доступа к ollama:\n{OLLAMA_URL}")

Корнева папка проекта:
/home/wirekit69/ai_engineer_preproject
Папка с документами датасета:
/home/wirekit69/ai_engineer_preproject/ollama_docs
URL для доступа к ollama:
http://localhost:11434


In [3]:
try:
    search_engine = VecIdxSearch(
        data_dir=str(DOCS_DIR),
        index_factory="Flat",
        # Можно передать кастомный набор расширений, если нужно:
        extensions={'.md', '.mdx', '.txt'} 
    )
    print("Индекс успешно построен. Поиск готов.")
except ValueError as e:
    print(f"Ошибка при инициализации: {e}")
    # Если здесь ошибка, поиск не сработает
    search_engine = None
print(f"Загружено документов: {search_engine.num_documents}")
print(f"Получено чанков: {search_engine.num_chunks}")   

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Индекс успешно построен. Поиск готов.
Загружено документов: 70
Получено чанков: 609


In [13]:
if search_engine:
    question = "Как сменить папку, в которой Ollama хранит модели?"
    
    # top_k=3 означает, что мы хотим получить 5 наиболее релевантных фрагментов
    results = search_engine.search(question=question, top_k=3)
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

#1  [openai-compatibility.mdx]  score=1.400
```python vision.py
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
) ...

#2  [openai-compatibility.mdx]  score=1.437
```javascript vision.js
import OpenAI from "openai";

const openai = new OpenAI({
  baseURL: "http://localhost:11434/v1/",
  apiKey: "ollama", // required but ignored
}); ...

#3  [chatgpt.mdx]  score=1.453
## Prerequisites

- Ollama v0.34.0 or newer

If ChatGPT is not installed, Ollama offers to download it during setup. ...



In [4]:
# Проверяем работоспособность сервера ollama и моделей
# Отправляем GET-запрос и получаем ответ
response = httpx.get(f"{OLLAMA_URL}/api/tags")
# Достаём из JSON-ответа список моделей
data = response.json()
models = data.get("models", [])
print(f"Найдено моделей: {len(models)}")
for model in models:
    print(f" - {model['name']}")

Найдено моделей: 5
 - qwen2.5-coder:1.5b
 - llama3.1:8b
 - qwen2.5-coder:7b
 - lfm3:1.7b
 - qwen2.5:1.5b


In [ ]:
for path in sorted(DOCS_DIR.rglob("*")):
    print(f"{str(path)} -> {path.name}")


In [ ]:

# ЗАГРУЖАЕМ ДОКУМЕНТЫ
documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

In [ ]:
# НАРЕЗАЕМ НА ЧАНКИ 
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

In [ ]:
# Посмотрим на несколько примеров чанков
for chunk in chunks[:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

In [5]:
# СВОДКА ПО ЧАНКАМ
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

source_counts = Counter(c["source"] for c in chunks)

top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")

Всего чанков: 609
Файлов с чанками: 70
Топ-3 файла по числу чанков:
  /home/wirekit69/ai_engineer_preproject/ollama_docs/api.md: 74 чанков
  /home/wirekit69/ai_engineer_preproject/ollama_docs/faq.mdx: 38 чанков
  /home/wirekit69/ai_engineer_preproject/ollama_docs/capabilities/tool-calling.mdx: 33 чанков


In [ ]:
file_path = DOCS_DIR / "api.md"
sample_text = file_path.read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    avg_length = sum(len(c) for c in chunks_sample) // len(chunks_sample) if chunks_sample else 0
    print(f"max_chars = {max_chars}\n-> чанков: {len(chunks_sample)}, средняя длина: {avg_length}")


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)

# Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

Загружаем модель эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Размер матрицы эмбеддингов: (609, 384)


In [8]:
print(f"Первый чанк: {chunks[0]['text'][:80]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Первый чанк: # Documentation

### Getting Started
* [Quickstart](https://docs.ollama.com/quic...
Его эмбеддинг (первые 10 чисел): [-0.06130737 -0.0551094  -0.03115822 -0.01005447  0.03433182  0.04836938
 -0.0826564   0.02022986 -0.00097113  0.04949645]
Длина вектора (норма): 1.0000


In [9]:


dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"В индексе {index.ntotal} векторов размерности {dim}") 

В индексе 609 векторов размерности 384


In [10]:


def vector_search(question: str, top_k: int = 3) -> list[dict]:
    """Возвращает top_k чанков, наиболее близких к запросу по смыслу."""
    # 1. Считаем эмбеддинг запроса
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)

    # 2. Ищем ближайших соседей
    scores, indices = index.search(query_emb, top_k)

    # 3. Собираем результат с метаданными
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

In [11]:
question = "Как сменить папку, в которой Ollama хранит модели?"


In [12]:
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

Запрос: Как сменить папку, в которой Ollama хранит модели?

#1  [/home/wirekit69/ai_engineer_preproject/ollama_docs/windows.mdx]  score=0.765
To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `OLLAMA_MODELS` in your user account.

1. Start the Settings (Windows 11) or Cont ...

#2  [/home/wirekit69/ai_engineer_preproject/ollama_docs/macos.mdx]  score=0.629
## Filesystem Requirements

The preferred method of installation is to mount the `ollama.dmg` and drag-and-drop the Ollama application to the system-wide `Applications` folder.  Upon startup, the Olla ...

#3  [/home/wirekit69/ai_engineer_preproject/ollama_docs/faq.mdx]  score=0.620
Alternatively, you can change the amount of time all models are loaded into memory by setting the `OLLAMA_KEEP_ALIVE` environment variable when starting the Ollama server. The `OLLAMA_KEEP_ALIVE` vari ...



In [ ]:
# --- Использование ---

# 1. Подготовка данных (один раз при старте приложения)
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# chunks = [...]
# index = faiss.IndexIDMap(...)

# 2. Создание экземпляра (инъекция зависимостей)
vector_db = VectorStore(
    embed_model_name=EMBED_MODEL,
    index=index,
    chunks=chunks
)

# 3. Выполнение поиска
results = vector_db.search("Как сменить папку, в которой Ollama хранит модели?", top_k=5)

In [ ]:
print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

In [ ]:
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    results = vector_db.search(q, top_k=5) 
    for r in results:
        print(f"  [{r['source']}]  score={r['score']:.3f}")
        print(f"  {r['text'][:150].strip()}...\n") 

